# ML-Powered Forecasting at Scale

This notebook demonstrates how **polars-ts** turns scikit-learn estimators into
production-grade forecasting systems. We cover:

| Concept | API |
|---|---|
| End-to-end pipeline | `ForecastPipeline` |
| Recursive (iterative) strategy | `RecursiveForecaster` |
| Direct (one-model-per-step) strategy | `DirectForecaster` |
| Global (cross-series) modelling | `GlobalForecaster` |
| Time-series cross-validation | `sliding_window_cv`, `rolling_origin_cv` |
| Explainability | `permutation_importance` |

**Data**: M4 Hourly competition dataset (5 series).

> **References**
> - Makridakis, S., Spiliotis, E., & Assimakopoulos, V. (2020). *The M4 Competition*. International Journal of Forecasting.
> - Hewamalage, H., Bergmeir, C., & Bandara, K. (2021). *Global Models for Time Series Forecasting: A Simulation Study*. Pattern Recognition.
> - Januschowski, T. et al. (2020). *Criteria for Classifying Forecasting Methods*. International Journal of Forecasting.

In [ ]:
# Install polars-timeseries if not already available
import importlib

if importlib.util.find_spec("polars_ts") is None:
    %pip install -q polars-timeseries[all]

## 1. Setup

In [ ]:
try:
    import hvplot.polars  # noqa
except ImportError:
    pass  # hvplot is optional for visualization
import polars as pl
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge

from polars_ts import (
    DirectForecaster,
    ForecastPipeline,
    GlobalForecaster,
    RecursiveForecaster,
    mae,
    permutation_importance,
    rmse,
    rolling_origin_cv,
    sliding_window_cv,
)

## 2. Data Loading (M4 Hourly)

We select five hourly series (`H1`--`H5`) from the M4 competition dataset and
split into train / test. The last 48 observations per series are held out
(matching the original M4 forecast horizon).

In [ ]:
df = (
    pl.scan_parquet("https://datasets-nixtla.s3.amazonaws.com/m4-hourly.parquet")
    .filter(pl.col("unique_id").is_in(["H1", "H2", "H3", "H4", "H5"]))
    .collect()
)

HORIZON = 48

train = df.group_by("unique_id").agg(pl.all().head(-HORIZON)).explode(pl.all().exclude("unique_id"))
test = df.group_by("unique_id").agg(pl.all().tail(HORIZON)).explode(pl.all().exclude("unique_id"))

print(f"Train: {train.shape}, Test: {test.shape}")
train.head()

## 3. ForecastPipeline -- End-to-End Forecasting

`ForecastPipeline` wraps a scikit-learn estimator and automates feature
engineering (lags, rolling statistics, calendar dummies) and multi-series
forecasting in a single object. Think of it as a batteries-included baseline.

In [ ]:
pipeline = ForecastPipeline(
    estimator=Ridge(),
    lags=[1, 2, 3, 24],
    rolling_windows=[24],
    rolling_aggs=["mean", "std"],
    calendar=["hour", "day_of_week"],
)

pipeline.fit(train)
preds_pipeline = pipeline.predict(train, h=HORIZON)

preds_pipeline.head(10)

In [ ]:
# Quick visual check for one series
sid = "H1"
actual = test.filter(pl.col("unique_id") == sid)
forecast = preds_pipeline.filter(pl.col("unique_id") == sid)

(
    actual.hvplot.line(x="ds", y="y", label="Actual", color="black", width=780, height=320)
    * forecast.hvplot.line(x="ds", y="y_hat", label="Pipeline (Ridge)", color="#1f77b4")
).opts(title=f"ForecastPipeline -- {sid}", ylabel="y")

## 4. RecursiveForecaster

The **recursive** (or *iterative*) strategy trains a single model on one-step
ahead targets, then feeds its own predictions back as inputs to generate
multi-step forecasts. This is simple and parameter-efficient but can
accumulate errors over longer horizons.

In [ ]:
rec = RecursiveForecaster(
    estimator=GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42),
    lags=[1, 2, 3, 24],
)

rec.fit(train)
preds_recursive = rec.predict(train, h=HORIZON)

preds_recursive.head(10)

## 5. DirectForecaster

The **direct** strategy trains one independent model for *each* forecast step.
This avoids error accumulation but increases training cost linearly with the
horizon `h`.

In [ ]:
direct = DirectForecaster(
    estimator_factory=lambda: Ridge(),
    lags=[1, 2, 3, 24],
    h=HORIZON,
)

direct.fit(train)
preds_direct = direct.predict(train)

preds_direct.head(10)

## 6. GlobalForecaster

A **global** model pools data from all series into a single training set,
optionally encoding the series identity. This is the key insight behind modern
ML forecasting at scale: shared patterns across series give the model more
signal, especially when individual series are short.

In [ ]:
glob = GlobalForecaster(
    estimator=Ridge(),
    lags=[1, 2, 3, 24],
    rolling_windows=[24],
    encode_id="onehot",
)

glob.fit(train)
preds_global = glob.predict(train, h=HORIZON)

preds_global.head(10)

## 7. Cross-Validation

Evaluating forecasters on a single train/test split is fragile.
**polars-ts** ships two time-aware CV strategies:

| Strategy | Behaviour |
|---|---|
| `sliding_window_cv` | Fixed-size training window slides forward |
| `rolling_origin_cv` | Training set grows; only the forecast origin advances |

In [ ]:
# -- Sliding-window CV --
sw_scores = []
for fold, (tr, te) in enumerate(sliding_window_cv(df, n_splits=5, train_size=200, horizon=HORIZON)):
    pipeline.fit(tr)
    preds = pipeline.predict(tr, h=HORIZON)
    merged = te.join(preds, on=["unique_id", "ds"])
    sw_scores.append(
        {"fold": fold, "mae": mae(merged, "y", "y_hat"), "rmse": rmse(merged, "y", "y_hat"), "cv": "sliding_window"}
    )

# -- Rolling-origin CV --
ro_scores = []
for fold, (tr, te) in enumerate(rolling_origin_cv(df, n_splits=5, horizon=HORIZON)):
    pipeline.fit(tr)
    preds = pipeline.predict(tr, h=HORIZON)
    merged = te.join(preds, on=["unique_id", "ds"])
    ro_scores.append(
        {"fold": fold, "mae": mae(merged, "y", "y_hat"), "rmse": rmse(merged, "y", "y_hat"), "cv": "rolling_origin"}
    )

cv_results = pl.DataFrame(sw_scores + ro_scores)
cv_results

## 8. Feature Importance

Which features actually drive the forecast? `permutation_importance` shuffles
each feature column and measures the increase in prediction error, giving a
model-agnostic importance ranking.

In [ ]:
feature_cols = [c for c in pipeline.feature_names_ if c not in ("unique_id", "ds", "y")]

imp = permutation_importance(test, pipeline, feature_cols=feature_cols)

imp.sort("importance_mean", descending=True).hvplot.barh(
    y="feature",
    x="importance_mean",
    xerr="importance_std",
    color="#4c72b0",
    title="Permutation Importance (ForecastPipeline)",
    width=700,
    height=350,
    xlabel="Mean importance",
    ylabel="",
)

## 9. Model Comparison

Let us bring everything together and compare all four forecasters on the
held-out test set.

In [ ]:
results = []
for name, preds in [
    ("Pipeline (Ridge)", preds_pipeline),
    ("Recursive (GBR)", preds_recursive),
    ("Direct (Ridge)", preds_direct),
    ("Global (Ridge)", preds_global),
]:
    merged = test.join(preds, on=["unique_id", "ds"])
    results.append(
        {
            "model": name,
            "MAE": round(mae(merged, "y", "y_hat"), 2),
            "RMSE": round(rmse(merged, "y", "y_hat"), 2),
        }
    )

comparison = pl.DataFrame(results)
print(comparison)

comparison.hvplot.bar(
    x="model",
    y=["MAE", "RMSE"],
    rot=15,
    title="Test-set Error by Forecasting Strategy",
    width=700,
    height=350,
    ylabel="Error",
    color=["#4c72b0", "#dd8452"],
)

## Summary

| What we learned | Detail |
|---|---|
| **ForecastPipeline** | One-liner baseline: lags + rolling stats + calendar features |
| **RecursiveForecaster** | Feed-back loop; simple but error compounds over long horizons |
| **DirectForecaster** | One model per step; no error accumulation, higher training cost |
| **GlobalForecaster** | Pool all series; best when individual histories are short |
| **Cross-validation** | `sliding_window_cv` for stationarity-aware eval; `rolling_origin_cv` for expanding-window eval |
| **Permutation importance** | Model-agnostic feature ranking -- sanity-check your pipeline |

**Next steps**: swap in more powerful estimators (LightGBM, XGBoost),
add exogenous regressors, or explore conformal prediction intervals.